# Phenotypic bridges and fitness-valley crossing

A population fixed at genotype 0 must reach genotype 2 through the low-fitness
genotype 1. Genotype 1 expresses the *high*-fitness phenotype with probability
$\phi$ &mdash; the phenotypic bridge. Larger $\phi$ crosses the valley faster,
even though the mutation rate is unchanged.

Runs from `results/bridge/summary.npz` (mean $\pm$ SEM over 100 trials per
condition, recorded every 10th dilution cycle).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from propgen import load_summary
from propgen.plotting import set_paper_style

set_paper_style()
FIGDIR = REPO / "figures"
FIGDIR.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO / "experiments" / "bridge"))
from analysis import FAR_SIDE_PAIR, fit_tau, tau_vs_phi

summary = load_summary(REPO / "results" / "bridge" / "summary.npz")
summary

The summary's stored coordinate is `bridge_mapping`; the bridge probability is
$\phi = 1 - \texttt{bridge\_mapping}$.

In [ ]:
PAIR_COLORS = {(0, 0): "blue", (1, 0): "green", (1, 1): "red", (2, 0): "purple"}
SELECTED = [(0, 0), (1, 0), (1, 1), (2, 0)]
GAMMA = 0.0002                      # deep valley
PHIS = [0.00, 0.05, 0.10, 0.15, 0.40]

fig, axes = plt.subplots(len(PHIS), 1, figsize=(6, 5 * len(PHIS)), sharex=True)

for ax, phi in zip(axes, PHIS):
    panel = summary.select(bridge_mapping=1 - phi, gamma=GAMMA)
    mean, sem, cycles = panel["mean"], panel["sem"], panel["cycles"]
    f_eq = panel["f_eq"].reshape(summary.shape)

    for g, p in SELECTED:
        color = PAIR_COLORS[(g, p)]
        ax.plot(cycles, mean[g, p], color=color, lw=2, alpha=0.8, label=f"$g$={g}, $p$={p}")
        ax.fill_between(cycles, mean[g, p] - sem[g, p], mean[g, p] + sem[g, p],
                        color=color, alpha=0.2)
        ax.axhline(f_eq[g, p], color=color, ls="--", lw=2, alpha=0.3)

    ax.set_title(rf"$\phi_1^{{(0)}}$ = {phi:.2f}", fontsize=18)
    ax.set_ylabel("Frequency")

axes[-1].set_xlabel("Dilution cycle")
axes[0].legend(fontsize=10, loc="best")
plt.tight_layout()

for ax in axes:
    for artist in ax.get_children():
        if hasattr(artist, "set_rasterized"):
            artist.set_rasterized(True)

plt.savefig(FIGDIR / "bridge_trajectories.pdf", dpi=300)
plt.show()

## Relaxation time

The approach of the far-side pair $(2,0)$ to its equilibrium is fitted to a
decaying exponential; $\tau$ is the resulting relaxation time in dilution
cycles. It falls by more than an order of magnitude as the bridge opens.

In [ ]:
plt.figure(figsize=(7, 5))
for gamma, marker in zip(sorted(summary.values("gamma")), ["o", "s", "^"]):
    phis, taus = tau_vs_phi(summary, gamma)
    plt.plot(phis, taus, marker=marker, lw=2, label=rf"$\gamma$ = {gamma}")

plt.yscale("log")
plt.xlabel(r"Bridge probability $\phi_1^{(0)}$")
plt.ylabel(r"Relaxation time $\tau$ (cycles)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGDIR / "bridge_tau_vs_phi.pdf")
plt.show()

phis, taus = tau_vs_phi(summary, 0.0002)
for phi, tau in list(zip(phis, taus))[:6]:
    print(f"  phi = {phi:.2f}   tau = {tau:10.0f} cycles")